Cell 1

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_log_error

BASE_DIR = Path(
    r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting"
)

ADVANCED_DIR = BASE_DIR / "data" / "advanced"

train = pd.read_csv(
    ADVANCED_DIR /
    "train_expanded_ewma_seasonal.csv"
)

train["date"] = pd.to_datetime(
    train["date"]
)

print(train.shape)

(3000888, 57)


Cell 2

In [2]:
target = "sales"

features = [
    "store_nbr", "family", "onpromotion",

    "year", "month", "day", "quarter", "week",
    "day_of_week", "is_weekend",

    "city", "state", "type", "cluster",

    "dcoilwtico",
    "is_payday",
    "is_national_holiday",
    "earthquake_period",

    "sales_lag_1",
    "sales_lag_2",
    "sales_lag_3",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_21",
    "sales_lag_28",
    "sales_lag_56",

    "rolling_mean_3",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_mean_56",

    "rolling_median_7",
    "rolling_median_14",
    "rolling_median_28",

    "rolling_min_7",
    "rolling_min_14",
    "rolling_min_28",

    "rolling_max_7",
    "rolling_max_14",
    "rolling_max_28",

    "rolling_std_7",
    "rolling_std_14",
    "rolling_std_28",
    "rolling_std_56",

    "promo_rolling_mean_7",
    "promo_rolling_mean_14",
    "promo_rolling_mean_28",

    "ewma_03",
    "ewma_05",
    "ewma_07",

    "sales_lag_364",
    "sales_lag_365",
    "sales_lag_371",
    "rolling_mean_364_7"
]

X = train[features]

y = np.log1p(
    train[target]
)

print(X.shape)

(3000888, 54)


Cell 3

In [3]:
X = pd.get_dummies(X, drop_first=True)

split_date = "2017-01-01"

train_mask = train["date"] < split_date
valid_mask = train["date"] >= split_date

X_train = X.loc[train_mask]
X_valid = X.loc[valid_mask]

y_train = y.loc[train_mask]

y_valid_original = train.loc[
    valid_mask,
    "sales"
]

Cell 4 - Model A

In [4]:
model_a = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_a.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=8, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

Cell 5 - Model B

In [5]:
model_b = XGBRegressor(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_b.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.03, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=700, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

Cell 6 - Model C

In [6]:
model_c = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_c.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.03, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=300, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

Cell 7 - Ensemble prediction

In [7]:
pred_a = model_a.predict(X_valid)
pred_b = model_b.predict(X_valid)
pred_c = model_c.predict(X_valid)

ensemble_log = (
    0.60 * pred_a +
    0.25 * pred_b +
    0.15 * pred_c
)

ensemble_preds = np.expm1(ensemble_log)
ensemble_preds = np.maximum(ensemble_preds, 0)

rmsle = np.sqrt(
    mean_squared_log_error(
        y_valid_original,
        ensemble_preds
    )
)

print("Ensemble RMSLE:", rmsle)

Ensemble RMSLE: 0.43160206347390717
